In [1]:
# %pip install pytest

In [2]:
# Prepare the Evaluation Dataset
# Ragas requires a Dataset object. Create a small "Golden Set" of questions you know the answers to 
# # (Ground Truth) and let your RAG system generate the rest.

reference_data = [
  {
    "question": "What is the company's policy on remote work?", 
    "ground_truth": "Remote work is allowed up to 3 days per week.", #Expected llm generated answer
    "context": "Remote work is allowed up to 3 days per week." #Expected retrieved context
  }
]
question = reference_data[0]['question']
ground_truth = reference_data[0]['ground_truth']
context = reference_data[0]['context']
print (f"question: {question}")
print (f"ground_truth: {ground_truth}")
print (f"context: {context}")

question: What is the company's policy on remote work?
ground_truth: Remote work is allowed up to 3 days per week.
context: Remote work is allowed up to 3 days per week.


In [3]:
# Retrieve context from Milvus DB

from milvus_chatbot_with_rag import retrieve_similiar_contexts, generate_answer

def perform_retrieval(question):

    retrieved_context = retrieve_similiar_contexts(question, "policy_docs_collection", 1)[0]['content']
    print (f"perform_retrieval.retrieved_context: {retrieved_context}")
    return retrieved_context

# Generate answer using LLM

question = "What is the company's policy on remote work?"
context = perform_retrieval(question)
answer = generate_answer(question, context)
answer

Connected to Milvus on Zilliz Cloud
perform_retrieval.retrieved_context: Company internet must be used for work-related tasks only.


'The provided policy doesn’t address remote work. It only states that company internet must be used for work-related tasks only.'

In [6]:

import os
import json
from dotenv import load_dotenv
from datasets import Dataset
from ragas import evaluate
from ragas.metrics.collections import faithfulness, answer_correctness
from openai import OpenAI

# --- Load API Key ---
load_dotenv(override=True, dotenv_path="../.env")
my_api_key = os.getenv("OPENAI_API_KEY")

client = OpenAI(api_key=my_api_key)

# JSONL loading code 
reference_data = []
# Use 'r' for raw string to handle backslashes safely
with open(r'.\data\training_data.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        data = json.loads(line)
        # Extract user question and assistant answer from the 'messages' list
        user_msg = next(m['content'] for m in data['messages'] if m['role'] == 'user')
        assistant_msg = next(m['content'] for m in data['messages'] if m['role'] == 'assistant')
            
        reference_data.append({
            "question": user_msg,
            "ground_truth": assistant_msg
        })


# --- 3. Prepare Evaluation Data ---
# For this example, we'll evaluate the first item. 
# In a real test, loop through all rows in reference_data.
item = reference_data[0]
question = item['question']
ground_truth = item['ground_truth']

# Get results from your system (ensure these functions are defined in your environment)
# retrieved_context should be a list of strings: ["chunk 1", "chunk 2"]
retrieved_context = perform_retrieval(question) 
llm_answer = generate_answer(question, retrieved_context)

# --- 4. Build Ragas Dataset ---
# 'contexts' must be a list of lists. 'reference' is the modern key for ground truth.
dataset_dict = {
    "question": [question],
    "answer": [llm_answer],
    "contexts": [retrieved_context], 
    "reference": [ground_truth]      
}

ragas_dataset = Dataset.from_dict(dataset_dict)

# --- 5. Run Evaluation ---
results = evaluate(
    ragas_dataset,
    metrics=[faithfulness(), answer_correctness()]
)

print("\n--- Evaluation Results ---")
print(results)

Connected to Milvus on Zilliz Cloud
perform_retrieval.retrieved_context: Employees can take an hour break.


TypeError: 'module' object is not callable

In [ ]:
results = evaluate(
    dataset=ragas_dataset,
    metrics=[faithfulness]  # just one metric
)


print("LLM Generation Evaluation Results:")
results.to_pandas()

In [ ]:
from ragas.llms.base import llm_factory
from ragas import evaluate
from ragas.metrics import answer_correctness

# Create the modern LLM wrapper
llm = llm_factory("gpt-4o-mini")

# Run evaluation
results = evaluate(
    dataset=ragas_dataset,
    metrics=[answer_correctness],
    llm=llm
)

print("LLM Generation Evaluation Results:")
print(results.to_pandas())